In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr
from lifelines import KaplanMeierFitter

ROOT = Path("data")
RAW_ROOT = ROOT / "raw"
BASE_DIR = ROOT / "models" / "baseline_pinned"
AUG_DIR  = ROOT / "models" / "augmented"
FUZZY_DIR = ROOT / "fuzzy"
CAL_DIR = ROOT / "eval" / "calibration_full"
CAL_DIR.mkdir(parents=True, exist_ok=True)

with open(ROOT / "processed" / "task_registry.json") as f:
    TASKS = json.load(f)
with open(ROOT / "processed" / "feature_list.json") as f:
    FEATURES = json.load(f)["features"]

FUZZY_COLS = ["mu_Safe", "mu_Watch", "mu_Danger", "mu_Critical", "interval_width"]
HORIZONS_DAYS = [7, 30, 90]
HORIZONS_SEC = [d * 86400 for d in HORIZONS_DAYS]

HOLDOUT_START = pd.Timestamp("2024-04-01").timestamp()
HOLDOUT_END   = pd.Timestamp("2024-07-02").timestamp()

base_models = list(BASE_DIR.glob("*.ubj"))
aug_models  = list(AUG_DIR.glob("*.ubj"))
print(f"baseline models saved: {len(base_models)}")
print(f"augmented models saved: {len(aug_models)}")

baseline models saved: 16
augmented models saved: 16


In [2]:
if len(base_models) == 0:
    print("WARNING: baseline models not saved. Need to retrain for Breslow.")
    print("Options: (a) retrain here, (b) skip Breslow and use decile calibration only")
if len(aug_models) == 0:
    print("WARNING: augmented models not saved. Need to retrain.")

In [5]:
from sklearn.preprocessing import LabelEncoder

CATEGORICAL = [c for c in ["userCoinTypeMode", "userReserveMode"] if c in FEATURES]
AUG_FEATURES = FEATURES + FUZZY_COLS

def load_task(task, with_fuzzy):
    tid = task["task_id"]
    cols = FEATURES + CATEGORICAL + ["timeDiff", "status", "timestamp"]
    cols = list(dict.fromkeys(cols))
    df = pd.read_csv(RAW_ROOT / task["train_path"].replace("\\", "/"), usecols=cols)
    for c in CATEGORICAL:
        df[c] = LabelEncoder().fit_transform(df[c].astype(str))
    df[FEATURES] = df[FEATURES].fillna(0)
    if with_fuzzy:
        fz = pd.read_parquet(FUZZY_DIR / f"{tid}_memberships.parquet")
        assert len(fz) == len(df)
        for col in FUZZY_COLS:
            df[col] = fz[col].values
    fit = df[df["timestamp"] < HOLDOUT_START].reset_index(drop=True)
    hold = df[(df["timestamp"] >= HOLDOUT_START) & (df["timestamp"] <= HOLDOUT_END)].reset_index(drop=True)
    return fit, hold

def breslow_baseline(times, events, risks):
    """Vectorized Breslow baseline, numerically stable."""
    order = np.argsort(times)
    t = np.asarray(times, dtype=float)[order]
    e = np.asarray(events, dtype=int)[order]
    r = np.asarray(risks, dtype=float)[order]
    r_max = float(r.max())  # shift for stability
    exp_r = np.exp(r - r_max)
    cum_from_right = np.cumsum(exp_r[::-1])[::-1]  # risk-set sums
    event_mask = e == 1
    event_times_sorted = t[event_mask]
    unique_et, counts = np.unique(event_times_sorted, return_counts=True)
    first_idx = np.searchsorted(t, unique_et)
    risk_set_sum = cum_from_right[first_idx]
    h = counts / np.maximum(risk_set_sum, 1e-12)
    H0 = np.cumsum(h)
    return unique_et, H0, r_max

def surv_at_horizon(event_times, H0, horizon, risk, r_max):
    idx = np.searchsorted(event_times, horizon, side='right') - 1
    H0_h = H0[idx] if idx >= 0 else 0.0
    risk_shifted = np.clip(risk - r_max, -30, 30)  # cap for safety
    return np.exp(-H0_h * np.exp(risk_shifted))

def km_censoring(y_time, y_status):
    """KM estimator of censoring distribution G(t). Status flipped."""
    kmf = KaplanMeierFitter()
    kmf.fit(y_time, event_observed=(1 - y_status))
    return kmf

def ipcw_brier(y_time, y_status, surv_prob, t, kmf_cens):
    """IPCW Brier at horizon t."""
    pred_event = 1 - surv_prob
    G_t = float(kmf_cens.survival_function_at_times(t).iloc[0]) or 1e-6
    y_at_t = (y_time <= t) & (y_status == 1)
    survived_past = y_time > t
    censored_before = (y_time <= t) & (y_status == 0)
    G_i = kmf_cens.survival_function_at_times(np.minimum(y_time, t)).values
    G_i = np.maximum(G_i, 1e-6)
    w = np.zeros(len(y_time))
    w[y_at_t] = 1.0 / G_i[y_at_t]
    w[survived_past] = 1.0 / G_t
    w[censored_before] = 0.0
    sq = (y_at_t.astype(float) - pred_event) ** 2
    return (w * sq).sum() / len(y_time)

print("helpers ready")

helpers ready


In [7]:
rows = []
for i, task in enumerate(TASKS, 1):
    tid = task["task_id"]
    print(f"[{i}/{len(TASKS)}] {tid}", flush=True)

    fit_b, hold_b = load_task(task, with_fuzzy=False)
    fit_a, hold_a = load_task(task, with_fuzzy=True)

    bst_b = xgb.Booster(); bst_b.load_model(str(BASE_DIR / f"{tid}.ubj"))
    bst_a = xgb.Booster(); bst_a.load_model(str(AUG_DIR  / f"{tid}.ubj"))

    r_fit_b = bst_b.predict(xgb.DMatrix(fit_b[FEATURES].values))
    r_fit_a = bst_a.predict(xgb.DMatrix(fit_a[AUG_FEATURES].values))
    r_hold_b = bst_b.predict(xgb.DMatrix(hold_b[FEATURES].values))
    r_hold_a = bst_a.predict(xgb.DMatrix(hold_a[AUG_FEATURES].values))

    et_b, H0_b, rmax_b = breslow_baseline(fit_b["timeDiff"].values, fit_b["status"].values, r_fit_b)
    et_a, H0_a, rmax_a = breslow_baseline(fit_a["timeDiff"].values, fit_a["status"].values, r_fit_a)
    kmf_cens = km_censoring(hold_b["timeDiff"].values, hold_b["status"].values)

    brier_b, brier_a = {}, {}
    for h_days, h_sec in zip(HORIZONS_DAYS, HORIZONS_SEC):
        S_b = surv_at_horizon(et_b, H0_b, h_sec, r_hold_b, rmax_b)
        S_a = surv_at_horizon(et_a, H0_a, h_sec, r_hold_a, rmax_a)
        brier_b[h_days] = ipcw_brier(hold_b["timeDiff"].values, hold_b["status"].values, S_b, h_sec, kmf_cens)
        brier_a[h_days] = ipcw_brier(hold_a["timeDiff"].values, hold_a["status"].values, S_a, h_sec, kmf_cens)

    # Integrated Brier Score (trapezoidal)
    ibs_b = np.trapezoid([brier_b[h] for h in HORIZONS_DAYS], HORIZONS_DAYS) / (HORIZONS_DAYS[-1] - HORIZONS_DAYS[0])
    ibs_a = np.trapezoid([brier_a[h] for h in HORIZONS_DAYS], HORIZONS_DAYS) / (HORIZONS_DAYS[-1] - HORIZONS_DAYS[0])
    print(f"       IBS base={ibs_b:.4f}  aug={ibs_a:.4f}  delta={ibs_a-ibs_b:+.4f}", flush=True)
    rows.append({
        "task_id": tid,
        "brier7_base": brier_b[7], "brier7_aug": brier_a[7],
        "brier30_base": brier_b[30], "brier30_aug": brier_a[30],
        "brier90_base": brier_b[90], "brier90_aug": brier_a[90],
        "ibs_base": ibs_b, "ibs_aug": ibs_a,
        "ibs_delta": ibs_a - ibs_b,
    })

cal_df = pd.DataFrame(rows)
print()
print(cal_df.round(4).to_string(index=False))
cal_df.to_csv(CAL_DIR / "brier_results.csv", index=False)
print(f"\nmean IBS delta (aug - base): {cal_df['ibs_delta'].mean():+.4f}")
print(f"aug better on: {(cal_df['ibs_delta'] < 0).sum()}/{len(cal_df)} tasks")

[1/16] Borrow_Deposit
       IBS base=0.6284  aug=0.6284  delta=+0.0000
[2/16] Borrow_Repay
       IBS base=0.1635  aug=0.1635  delta=+0.0000
[3/16] Borrow_Withdraw
       IBS base=0.2882  aug=0.7118  delta=+0.4236
[4/16] Borrow_Liquidated
       IBS base=0.9877  aug=0.9877  delta=+0.0000
[5/16] Deposit_Borrow
       IBS base=0.9305  aug=0.9305  delta=+0.0000
[6/16] Deposit_Repay
       IBS base=0.9314  aug=0.9314  delta=+0.0000
[7/16] Deposit_Withdraw
       IBS base=0.6153  aug=0.6153  delta=+0.0000
[8/16] Deposit_Liquidated
       IBS base=0.9815  aug=0.9815  delta=+0.0000
[9/16] Repay_Borrow
       IBS base=0.2766  aug=0.2766  delta=+0.0000
[10/16] Repay_Deposit
       IBS base=0.6433  aug=0.6433  delta=+0.0000
[11/16] Repay_Withdraw
       IBS base=0.2190  aug=0.2299  delta=+0.0109
[12/16] Repay_Liquidated
       IBS base=0.9878  aug=0.9878  delta=+0.0000
[13/16] Withdraw_Borrow
       IBS base=0.9810  aug=0.9810  delta=+0.0000
[14/16] Withdraw_Deposit
       IBS base=0.3958  aug=

In [8]:
unc_rows = []
for i, task in enumerate(TASKS, 1):
    tid = task["task_id"]
    _, hold_a = load_task(task, with_fuzzy=True)
    bst_a = xgb.Booster(); bst_a.load_model(str(AUG_DIR / f"{tid}.ubj"))
    r_hold = bst_a.predict(xgb.DMatrix(hold_a[AUG_FEATURES].values))

    # At median horizon (30 days)
    fit_a, _ = load_task(task, with_fuzzy=True)
    r_fit = bst_a.predict(xgb.DMatrix(fit_a[AUG_FEATURES].values))
    et, H0, rmax = breslow_baseline(fit_a["timeDiff"].values, fit_a["status"].values, r_fit)
    S30 = surv_at_horizon(et, H0, 30 * 86400, r_hold, rmax)
    pred_event = 1 - S30
    y_label = ((hold_a["timeDiff"].values <= 30*86400) & (hold_a["status"].values == 1)).astype(float)
    sq_err = (y_label - pred_event) ** 2

    width = hold_a["interval_width"].values
    rho, pval = spearmanr(width, sq_err)

    # 90% PI coverage: interval = [pred_event - width/2, pred_event + width/2]
    w_norm = (width - width.min()) / (width.max() - width.min() + 1e-9)  # normalized to [0,1]
    pi_lo = np.clip(pred_event - 0.5 * w_norm, 0, 1)
    pi_hi = np.clip(pred_event + 0.5 * w_norm, 0, 1)
    valid = (hold_a["timeDiff"].values > 30*86400) | ((hold_a["timeDiff"].values <= 30*86400) & (hold_a["status"].values == 1))
    covered = (y_label >= pi_lo) & (y_label <= pi_hi)
    coverage = covered[valid].mean()

    print(f"[{i}/{len(TASKS)}] {tid}  spearman(width, err)={rho:+.3f} p={pval:.3g}  coverage={coverage:.3f}", flush=True)
    unc_rows.append({
        "task_id": tid,
        "spearman_width_err": rho,
        "spearman_p": pval,
        "coverage_90pi": coverage,
    })

unc_df = pd.DataFrame(unc_rows)
print()
print(unc_df.round(4).to_string(index=False))
unc_df.to_csv(CAL_DIR / "uncertainty_results.csv", index=False)
print(f"\nmean spearman: {unc_df['spearman_width_err'].mean():+.3f}")
print(f"spearman > 0 and p < 0.05: {((unc_df['spearman_width_err'] > 0) & (unc_df['spearman_p'] < 0.05)).sum()}/{len(unc_df)}")
print(f"mean coverage (nominal 90%): {unc_df['coverage_90pi'].mean():.3f}")

[1/16] Borrow_Deposit  spearman(width, err)=+0.075 p=1.2e-122  coverage=0.370
[2/16] Borrow_Repay  spearman(width, err)=-0.058 p=5.43e-67  coverage=0.827
[3/16] Borrow_Withdraw  spearman(width, err)=+0.032 p=6.86e-24  coverage=0.251
[4/16] Borrow_Liquidated  spearman(width, err)=+0.024 p=9.96e-14  coverage=0.010
[5/16] Deposit_Borrow  spearman(width, err)=+0.020 p=7.1e-41  coverage=0.067
[6/16] Deposit_Repay  spearman(width, err)=+0.019 p=4e-37  coverage=0.066
[7/16] Deposit_Withdraw  spearman(width, err)=-0.197 p=0  coverage=0.374
[8/16] Deposit_Liquidated  spearman(width, err)=+0.045 p=8.85e-194  coverage=0.015
[9/16] Repay_Borrow  spearman(width, err)=-0.119 p=4.83e-261  coverage=0.719
[10/16] Repay_Deposit  spearman(width, err)=+0.082 p=1.66e-138  coverage=0.356
[11/16] Repay_Withdraw  spearman(width, err)=+0.047 p=4.65e-48  coverage=0.713
[12/16] Repay_Liquidated  spearman(width, err)=+0.041 p=3.07e-36  coverage=0.010
[13/16] Withdraw_Borrow  spearman(width, err)=+0.046 p=6.16e-21

In [10]:

print("FUZZY CALIBRATION VERDICT")

brier_wins = (cal_df["ibs_delta"] < -0.005).sum()
brier_losses = (cal_df["ibs_delta"] > 0.005).sum()
cov_closer = (np.abs(unc_df["coverage_90pi"] - 0.90) < 0.10).sum()
unc_sig_pos = ((unc_df["spearman_width_err"] > 0) & (unc_df["spearman_p"] < 0.05)).sum()

print(f"\nIBS: aug better than base by >0.005 on {brier_wins}/16 tasks")
print(f"IBS: aug worse than base by >0.005 on {brier_losses}/16 tasks")
print(f"PI coverage within 10% of nominal on {cov_closer}/16 tasks")
print(f"Width-error correlation positive and significant on {unc_sig_pos}/16 tasks")

print(f"\nMean IBS delta: {cal_df['ibs_delta'].mean():+.4f}")
print(f"Mean coverage: {unc_df['coverage_90pi'].mean():.3f}")
print(f"Mean width-error correlation: {unc_df['spearman_width_err'].mean():+.3f}")

print("\npre-registered decision:")
if brier_wins >= 10 and unc_sig_pos > unc_df.shape[0] / 2:
    print("  POSITIVE: fuzzy improves calibration AND uncertainty signal is informative")
elif brier_losses >= 10:
    print("  NEGATIVE: fuzzy worsens calibration on majority of tasks")
else:
    print("  NULL: no decisive calibration improvement from fuzzy")

FUZZY CALIBRATION VERDICT

IBS: aug better than base by >0.005 on 1/16 tasks
IBS: aug worse than base by >0.005 on 2/16 tasks
PI coverage within 10% of nominal on 1/16 tasks
Width-error correlation positive and significant on 12/16 tasks

Mean IBS delta: +0.0189
Mean coverage: 0.276
Mean width-error correlation: +0.007

pre-registered decision:
  NULL: no decisive calibration improvement from fuzzy
